<a href="https://colab.research.google.com/github/Arthur-Holanda/Arthur-Holanda/blob/main/Tarefa_U1_IA_Agentica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avaliação da Unidade I - Agente Financeiro Pessoal

**Autores:** ARTHUR FERREIRA DE HOLANDA e ENZO ARAÚJO DE SOUZA E OLIVEIRA

---

## 1. Visão Geral do Projeto

Este notebook contém a implementação de um agente capaz de resolver uma **tarefa verificável**, desenvolvido exclusivamente com a biblioteca `agentkit`. A entrega consiste neste único notebook, que foi projetado para executar do início ao fim sem intervenção manual.

O nosso agente atua como um **Assistente Financeiro Pessoal**. Ele é responsável por interpretar entradas em linguagem natural, registrar ou alterar transações (entradas e saídas) e gerar balanços exatos do estado financeiro atual. Com base na solicitação do usuário, o modelo decide dinamicamente quais ferramentas utilizar, a ordem de execução e o momento correto de encerrar o processamento.

### 1.1 Por que a tarefa é verificável?

O projeto exige a resolução de uma tarefa para a qual seja possível determinar objetivamente se a resposta gerada está correta ou incorreta. A gestão financeira encaixa-se perfeitamente nisso. Dado um estado financeiro inicial e uma série de instruções lógicas (como o recebimento de um salário ou o estorno de uma compra), métricas como o saldo final e os totais de categorias devem obrigatoriamente corresponder a um valor matemático exato, não havendo margem para ambiguidade no teste.

## 2. Principais Decisões de Projeto

O prompt e o contexto do agente foram projetados pela nossa dupla de acordo com a especificidade da tarefa financeira escolhida.

### 2.1 Ferramentas Desenvolvidas

Para atender ao requisito de possuir pelo menos três ferramentas desenvolvidas pela dupla utilizando o decorador `@tool`, criamos as seguintes funções:

* **`registrar_transacao`**: Unifica a adição de receitas e despesas, processando dados como valor, categoria, descrição e eventual acréscimo percentual (juros/multa) da movimentação.
* **`consultar_historico`**: Permite que o agente busque informações na memória de transações antes de tomar uma decisão (como encontrar o `id` exato de uma compra específica antes de efetuar o seu estorno).
* **`estornar_transacao`**: Estorna, total ou parcialmente, uma saída já registrada, devolvendo o valor ao saldo e descontando-o da categoria original de forma retroativa.
* **`gerar_balanco_financeiro`**: Calcula saldos totais, agrega os gastos por categoria e formata o resultado em JSON para validação.

A lógica construída permite a execução de tarefas que envolvem múltiplas etapas, garantindo que o agente exija o uso de mais de uma ferramenta em conjunto para resolver os casos propostos.

### 2.2 Mecanismos Utilizados

O agente implementa dois dos mecanismos avançados exigidos, garantindo que eles possuam uma função real na solução:

* **Memória / Estado:** O agente mantém um registro ativo do histórico de caixa. O estado é injetado dinamicamente no contexto da conversa por meio de *tags* a cada novo turno. Isso é essencial para que o modelo consiga processar sequências de interações complexas ao longo do tempo.
* **Saída Estruturada:** A entrega final retorna os dados em um formato rígido e estruturado, utilizando o contrato Pydantic `RelatorioFinanceiro` (gerado via `generate_structured`). Este mecanismo extrai o saldo final e os totais das categorias, possibilitando que a tarefa seja objetivamente verificada no código de teste pelo sistema avaliador.

## 3. Metodologia de Testes

O notebook implementa **dez casos de teste** visando cobrir situações variadas da tarefa. A complexidade dos testes não se baseia no tamanho massivo dos dados, mas sim na dificuldade lógica das instruções (operações que exigem consultas, estornos e interpretação matemática).

Pelo menos **três desses testes são suficientemente desafiadores para explorar os limites do agente**. Como exigido, para cada teste executado, será apresentado na tela a entrada, o resultado esperado, o resultado obtido e o status de aprovação. Por fim, apresentamos a taxa de acerto total e uma breve análise dos erros observados.

# 1. Modelagem do Prompt e Contexto

Nesta etapa, definimos o modelo de linguagem que atuará como o motor lógico do nosso agente e estruturamos a forma como ele interpreta as instruções financeiras.

### 1.1 Instanciação do Modelo (Groq via API)
O projeto exige que as chaves de API não fiquem expostas no código. Para cumprir esse requisito de segurança, a chave da API do Groq é lida da variável de ambiente `GROQ_API_KEY` e não aparece no notebook.

Utilizaremos a classe `LLMAPI` do `agentkit` para instanciar o modelo (por exemplo, `openai/gpt-oss-120b`), configurando a `base_url` para a interface compatível oferecida pelo Google. Além disso, a temperatura do modelo será fixada em `0.0`. Como finanças exigem exatidão e respostas determinísticas, uma temperatura zerada minimiza a criatividade do modelo, forçando-o a buscar a resposta exata e a confiar exclusivamente nas ferramentas matemáticas.

### 1.2 Anatomia do *System Prompt*
Para que o agente financeiro seja rigoroso e não se perca em conversas genéricas, o *prompt* não será um parágrafo corrido. Ele foi projetado como um *template* com componentes separados e reconhecíveis, facilitando a manutenção e a precisão do comportamento:

*   **Papel:** Define a persona (ex: "Você é um assistente financeiro pessoal estrito e analítico").
*   **Tarefa e Escopo:** Descreve a missão principal (ex: "Sua função é registrar receitas e despesas, executar estornos e emitir balanços financeiros precisos").
*   **Regras e Restrições:** Impõe os limites operacionais de domínio. Regras vitais incluem: "Nunca realize contas matemáticas de cabeça; dependa exclusivamente das ferramentas fornecidas" e "Classifique as despesas apenas nas categorias pré-aprovadas".

### 1.3 Delimitação e Proteção do Contexto
Em sistemas baseados em LLM, é comum que o modelo confunda a instrução do desenvolvedor com o texto digitado pelo usuário (o que pode gerar falhas ou *prompt injection*). Para garantir que o agente saiba exatamente o que é regra e o que é o dado da transação, aplicamos duas estratégias de delimitação de fronteiras:

1.  **Cabeçalhos Markdown:** Utilizamos formatação Markdown (como `# Papel`, `# Regras`, `# Ferramentas`) para separar os blocos lógicos da instrução de sistema. O modelo é extensivamente treinado nesse formato e compreende a hierarquia.
2.  **Tags de Estilo XML:** Todo dado dinâmico, como o histórico de transações inserido pela memória ou a entrada de texto atual do usuário, será empacotado dentro de tags claras (ex: `<historico_caixa> ... </historico_caixa>` ou `<comando_usuario> ... </comando_usuario>`). Isso isola os dados, forçando o modelo a tratar o conteúdo interno apenas como informação a ser processada, sem sobrescrever as regras do sistema.

In [ ]:
# Instalação do agentkit diretamente do repositório
!pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"


import os
import time
import json
from google.colab import userdata
from agentkit.model import LLMAPI
from agentkit.tools import tool
from agentkit.agent import Agent
from pydantic import BaseModel, Field

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# 1. Recupera a chave de API dos Secrets
#groq_key = userdata.get("GROQ_API_KEY")
groq_key = os.environ.get("GROQ_API_KEY")

#gemini_key = userdata.get("GOOGLE_API_KEY1")

# 2. Instancia o Gemini usando a interface compatível
# A temperatura é mantida em 0.0 para maximizar o determinismo, crucial para tarefas financeiras verificáveis.
#llm = LLMAPI(
#    model="gemini-2.5-flash",
#    api_key=gemini_key,
#    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
#    temperature=0.0,
#    max_tokens=4096,
#)
# 2. Instancia o modelo usando a interface da Groq
llm = LLMAPI(
    model="openai/gpt-oss-120b",
    api_key=groq_key,
    base_url="https://api.groq.com/openai/v1",
    temperature=0.0,
    max_tokens=4096,
)
print(f"Modelo instanciado: {llm.model}")

Modelo instanciado: openai/gpt-oss-120b


In [ ]:
# [CORREÇÃO 2] Lista fixa de categorias: citada no prompt e validada pela ferramenta registrar_transacao
CATEGORIAS = ["Salário", "Renda Extra", "Alimentação", "Transporte", "Moradia", "Saúde", "Contas", "Outros"]

# 3. Definição do System Prompt utilizando cabeçalhos Markdown para organizar os componentes
FINANCIAL_SYSTEM = f"""# Papel
Você é um assistente financeiro pessoal estrito e analítico.

# Tarefa e Escopo
Sua função é registrar receitas e despesas, executar estornos e consultar balanços financeiros exatos.

# Regras
- Classifique cada transação usando apenas estas categorias: {', '.join(CATEGORIAS)}.
- Nunca faça contas matemáticas de cabeça: dependa exclusivamente das ferramentas fornecidas.
- Juros ou multa: registre o valor original e informe o percentual em acrescimo_percentual (use 0 quando não houver).
- Se o usuário pedir para remover ou estornar um valor, consulte o histórico primeiro para obter o id da transação e use estornar_transacao.
- Antes da resposta final, chame gerar_balanco_financeiro.
- Responda de forma breve, educada e direta."""

# 4. Template do contexto com tags de estilo XML para isolar dados do usuário
CONTEXT_TEMPLATE = """<historico_caixa>
{history}
</historico_caixa>

<comando_usuario>
{user_input}
</comando_usuario>"""

def build_context(instruction: str, history: str, user_input: str) -> list[dict]:
    """Preenche o template do contexto e devolve as mensagens da chamada no formato esperado pelo modelo."""
    filled_context = CONTEXT_TEMPLATE.format(
        history=history if history else "(nenhuma transação registrada ainda)",
        user_input=user_input,
    )

    # O papel 'system' carrega as regras, o 'user' carrega os dados delimitados por tags XML
    return [
        {"role": "system", "content": instruction},
        {"role": "user", "content": filled_context},
    ]

# --- Teste Rápido do Prompt e Contexto ---
historico_teste = "Data: 2026-04-10 | Categoria: Salário | Valor: +5000.00"

# [CORREÇÃO APLICADA AQUI] Adicionamos uma instrução para o modelo não usar ferramentas neste teste específico
comando_teste = "Gastei 150 no supermercado hoje. Atualize. IMPORTANTE: Como isso é apenas um teste inicial de prompt, NÃO chame nenhuma ferramenta. Apenas explique em texto o que você faria e como classificaria essa transação."

mensagens = build_context(
    instruction=FINANCIAL_SYSTEM,
    history=historico_teste,
    user_input=comando_teste
)

# Invocando o modelo para confirmar que ele entendeu o papel (sem ferramentas ainda)
resposta = llm.invoke(mensagens)
print("\n--- Resposta do Teste de Prompt ---")
print(resposta)


--- Resposta do Teste de Prompt ---
Para registrar essa despesa, eu criaria uma nova entrada no caixa com as seguintes informações:

- **Data:** a data de hoje (2026‑04‑27).  
- **Categoria:** **Alimentação** – porque a compra foi feita em um supermercado, que se enquadra nos gastos de alimentação.  
- **Valor:** **‑150.00** – valores de despesa são registrados com sinal negativo.  

Depois de inserir essa linha no histórico, eu geraria o balanço financeiro atualizado para refletir o novo saldo. Como solicitado, não estou chamando nenhuma ferramenta neste momento; apenas descrevo o procedimento que seria executado.


## 2. Definição do Estado e Memória

Modelos de linguagem, por natureza, tratam cada chamada de forma isolada, não mantendo conhecimento sobre interações passadas a menos que essa informação seja fornecida no *prompt*. A gestão financeira exige que o assistente enxergue o histórico contínuo das transações, mantendo um estado que sobrevive entre chamadas simples.

Para resolver isso, implementamos uma **Memória de Curto Prazo (Estado do Agente)** baseada nas práticas do material de referência:
* **O Estado Central:** Criamos uma estrutura em formato de dicionário Python (`finance_state`) contendo o saldo em tempo real, o registro histórico de transações, uma agregação por categorias e o histórico das mensagens da sessão.
* **Fonte Única de Verdade:** Este dicionário atua como o núcleo da memória da sessão. Ele será alterado no lugar ("*in-place*") pelas ferramentas financeiras dinâmicas (que serão implementadas a seguir).
* **Injeção Dinâmica:** Projetamos uma função formatadora que lê esse estado atualizado e o transforma em uma representação textual. A cada novo turno da conversa, essa string é injetada no contexto, dentro das tags `<historico_caixa>` do `build_context`, permitindo que as decisões lógicas e matemáticas do modelo sejam ancoradas exclusivamente nos fatos financeiros exatos e atuais.

In [ ]:
# 1. Estrutura do Estado do Agente (Memória de Curto Prazo)
# O estado concentra todas as informações que precisam sobreviver ao longo da execução.
finance_state = {
    "saldo": 0.0,
    "transacoes": [],        # Guardará dicionários: [{'id': 0, 'tipo': '...', 'valor': 0.0, 'categoria': '...', 'descricao': '...'}]
    "categorias_resumo": {}, # Guardará totais por categoria: {'Alimentação': 150.0, 'Salário': 5000.0}
    "messages": []           # Opcional: para manter o histórico conversacional bruto, se necessário
}

# 2. Template de Injeção de Estado
# As chaves {saldo}, {transacoes_texto} e {categorias_texto} serão substituídas dinamicamente.
STATE_TEMPLATE = """--- ESTADO FINANCEIRO ATUAL ---
Saldo Atual: R$ {saldo:.2f}

Transações Registradas:
{transacoes_texto}

Resumo por Categoria:
{categorias_texto}
-------------------------------"""

# 3. Função Formatadora
def format_financial_state(state: dict) -> dict:
    """
    Lê o dicionário de estado atual e o injeta no gabarito do sistema.
    Devolve a mensagem pronta com a role 'system'.
    """
    # Formata a lista de transações para texto legível
    if not state["transacoes"]:
        transacoes_texto = "(nenhuma transação registrada)"
    else:
        transacoes_texto = "\n".join(
            f"- {t['tipo'].upper()}: R$ {t['valor']:.2f} | Categoria: {t['categoria']} ({t['descricao']})"
            for t in state["transacoes"]
        )

    # Formata a agregação de categorias
    if not state["categorias_resumo"]:
        categorias_texto = "(nenhum gasto categorizado)"
    else:
        categorias_texto = "\n".join(
            f"- {cat}: R$ {val:.2f}" for cat, val in state["categorias_resumo"].items()
        )

    # Preenche o template e retorna o bloco de sistema formatado
    content = STATE_TEMPLATE.format(
        saldo=state["saldo"],
        transacoes_texto=transacoes_texto,
        categorias_texto=categorias_texto
    )

    return {"role": "system", "content": content}

# --- Teste de Visualização do Estado ---
# Simulando a adição de dados à memória pelas ferramentas (que faremos na próxima etapa)
finance_state["saldo"] = 4850.00
finance_state["transacoes"].append({
    "data": "2026-04-10", "tipo": "entrada", "valor": 5000.00, "categoria": "Salário", "descricao": "Pagamento mensal"
})
finance_state["transacoes"].append({
    "data": "2026-04-11", "tipo": "saida", "valor": 150.00, "categoria": "Alimentação", "descricao": "Supermercado"
})
finance_state["categorias_resumo"]["Salário"] = 5000.00
finance_state["categorias_resumo"]["Alimentação"] = 150.00

# Imprimindo o prompt dinâmico gerado
mensagem_sistema = format_financial_state(finance_state)
print(mensagem_sistema["content"])

--- ESTADO FINANCEIRO ATUAL ---
Saldo Atual: R$ 4850.00

Transações Registradas:
- ENTRADA: R$ 5000.00 | Categoria: Salário (Pagamento mensal)
- SAIDA: R$ 150.00 | Categoria: Alimentação (Supermercado)

Resumo por Categoria:
- Salário: R$ 5000.00
- Alimentação: R$ 150.00
-------------------------------


## 3. Implementação das Ferramentas e Orquestração do Agente

Um modelo de linguagem produz apenas texto; calcular saldos com exatidão ou buscar itens no histórico requer execução fora dele, em código Python. Nesta etapa, conectamos a lógica de linguagem à execução de código através de **ferramentas**.

Para cumprir as exigências da avaliação, implementamos quatro ferramentas  desenvolvidas pela nossa dupla utilizando o decorador `@tool`:
1.  **`registrar_transacao`**: Unifica a adição de entradas e saídas, alterando dinamicamente a nossa estrutura de memória (`finance_state`).
2.  **`consultar_historico`**: Permite ao agente investigar o passado financeiro antes de tomar uma decisão (crucial para pedidos de estorno, por exemplo).
3.  **`estornar_transacao`**: Estorna uma saída já registrada a partir do seu `id`, devolvendo o valor ao saldo e descontando-o da categoria. Exige consultar o histórico antes.
4.  **`gerar_balanco_financeiro`**: Consolida o estado final em um formato legível por máquina (saldo, total gasto e gastos por categoria). Em seguida, o harness converte o atendimento no contrato Pydantic `RelatorioFinanceiro` via `generate_structured`, que é o nosso mecanismo de **Saída Estruturada** e permite a verificação objetiva no teste.
Por fim, instanciamos a classe `Agent` nativa do `agentkit`. Ela implementa o laço de raciocínio (ReAct), dando autonomia para o sistema decidir dinamicamente quais ferramentas utilizar, a ordem dessas chamadas e o momento de encerrar a execução para devolver a resposta final ao usuário.

In [ ]:
# ==========================================
# 1. DEFINIÇÃO DAS FERRAMENTAS (@tool)
# ==========================================
# O @tool do agentkit envia ao modelo só a PRIMEIRA linha da docstring,
# por isso as instruções de cada ferramenta foram concentradas nessa linha.

@tool
def registrar_transacao(tipo: str, valor: float, categoria: str, descricao: str, acrescimo_percentual: float) -> str:
    """Registra uma transação nova. tipo: 'entrada' ou 'saida'; valor: positivo e sem juros; acrescimo_percentual: juros/multa em % (0 se não houver); categoria: uma das categorias permitidas."""
    if tipo not in ["entrada", "saida"]:
        return "Erro: O tipo deve ser 'entrada' ou 'saida'."

    # Valida categoria e valor, e aplica juros/multa no código (e não "de cabeça")
    if categoria not in CATEGORIAS:
        return f"Erro: categoria inválida. Use uma de {CATEGORIAS}."
    if valor <= 0:
        return "Erro: O valor deve ser positivo."
    valor = round(valor * (1 + acrescimo_percentual / 100), 2)

    # Atualiza o histórico na memória
    # Cada transação ganha um id, usado por estornar_transacao
    id_transacao = len(finance_state["transacoes"])
    finance_state["transacoes"].append({
        "id": id_transacao,
        "tipo": tipo,
        "valor": valor,
        "categoria": categoria,
        "descricao": descricao
    })

    # Atualiza saldos e agrupamentos
    if tipo == "entrada":
        finance_state["saldo"] += valor
    else:
        finance_state["saldo"] -= valor
        finance_state["categorias_resumo"][categoria] = finance_state["categorias_resumo"].get(categoria, 0.0) + valor

    return f"Sucesso: {tipo} de R$ {valor:.2f} registrada na categoria '{categoria}' (id={id_transacao})."  # [CORREÇÃO 5] devolve o id

@tool
def consultar_historico() -> str:
    """Consulta o histórico completo de transações (com id, tipo, valor e categoria); use antes de estornos ou de cálculos baseados no passado."""
    if not finance_state["transacoes"]:
        return "Aviso: Nenhuma transação registrada no momento."

    # Devolve os dados em JSON para o modelo analisar como observação
    return json.dumps(finance_state["transacoes"], ensure_ascii=False)

# estorno de verdade. Antes o estorno só podia ser registrado como
# 'entrada', o que corrigia o saldo mas não descontava o valor da categoria original.
@tool
def estornar_transacao(id_transacao: int, valor: float) -> str:
    """Estorna, total ou parcialmente, uma saida já registrada; consulte o histórico antes para obter o id_transacao."""
    transacoes = finance_state["transacoes"]
    i = int(id_transacao)
    if not 0 <= i < len(transacoes) or transacoes[i]["tipo"] != "saida":
        return "Erro: id_transacao não corresponde a uma saída registrada. Consulte o histórico."

    ja_estornado = sum(t["valor"] for t in transacoes if t.get("estorno_de") == i)
    restante = transacoes[i]["valor"] - ja_estornado
    if not 0 < valor <= restante + 0.005:
        return f"Erro: valor inválido. O máximo estornável é R$ {restante:.2f}."

    categoria = transacoes[i]["categoria"]
    transacoes.append({
        "id": len(transacoes),
        "tipo": "estorno",
        "valor": valor,
        "categoria": categoria,
        "descricao": f"Estorno da transação {i}",
        "estorno_de": i
    })
    finance_state["saldo"] += valor
    finance_state["categorias_resumo"][categoria] -= valor
    return f"Sucesso: estorno de R$ {valor:.2f} aplicado à transação {i} ('{categoria}')."

@tool
def gerar_balanco_financeiro() -> str:
    """Gera o balanço final em JSON (saldo, total gasto e gastos por categoria); deve ser a ÚLTIMA ferramenta chamada antes da resposta final."""
    # Consolida os dados cruciais para validação objetiva
    balanco = {
        "saldo_final": round(finance_state["saldo"], 2),
        "total_gastos": round(sum(finance_state["categorias_resumo"].values()), 2),
        # Gastos por categoria, necessários para verificar total_alimentacao e total_transporte
        "gastos_por_categoria": {cat: round(val, 2) for cat, val in finance_state["categorias_resumo"].items()}
    }
    # Retorna o dicionário serializado para que o modelo entenda os campos rígidos
    return json.dumps(balanco, ensure_ascii=False)


# ==========================================
# 2. INSTANCIAÇÃO E ORQUESTRAÇÃO DO AGENTE
# ==========================================

# Instanciamos o Agente passando o modelo (llm configurado no passo 1) e as ferramentas
# max_steps=15 garante que o agente tenha turnos suficientes para tarefas de múltiplas etapas,
# mas corta execuções que entrarem em loop infinito.
agent = Agent(
    llm=llm,
    tools=[registrar_transacao, consultar_historico, estornar_transacao, gerar_balanco_financeiro],  # [CORREÇÃO 6]
    max_steps=15
)

print("Ferramentas carregadas e Agente Financeiro instanciado e pronto para uso!")

Ferramentas carregadas e Agente Financeiro instanciado e pronto para uso!


In [ ]:
# Contrato da Saída Estruturada
class RelatorioFinanceiro(BaseModel):
    saldo_final: float | None = Field(default=None, description="O saldo final após todas as operações.")
    total_gastos: float | None = Field(default=None, description="O total de todas as saídas, já descontados os estornos.")  # descrição alinhada ao teste 10
    total_alimentacao: float | None = Field(default=None, description="Total gasto especificamente na categoria Alimentação.")
    total_transporte: float | None = Field(default=None, description="Total gasto especificamente na categoria Transporte.")

def resetar_memoria():
    """Limpa o estado global do agente para que um teste não contamine o outro."""
    finance_state["saldo"] = 0.0
    finance_state["transacoes"].clear()
    finance_state["categorias_resumo"].clear()

# Saída estruturada de verdade: o atendimento do agente (incluindo o JSON devolvido por
# gerar_balanco_financeiro) é convertido no contrato RelatorioFinanceiro via generate_structured.
# Antes o RelatorioFinanceiro era definido mas nunca usado.
def extrair_relatorio(mensagens: list[dict]) -> dict:
    """Converte o atendimento do agente no contrato RelatorioFinanceiro e devolve um dicionário."""
    atendimento = "\n".join(
        f"{m['role']}: {m['content']}" for m in mensagens if m["role"] != "system" and m.get("content")
    )
    relatorio = llm.generate_structured(
        [
            {"role": "system", "content": "Extraia os valores finais do atendimento abaixo. Use null para valores que não aparecem."},
            {"role": "user", "content": f"<atendimento>\n{atendimento}\n</atendimento>"},
        ],
        RelatorioFinanceiro,
    )
    return relatorio.model_dump()

## 4. Estruturação dos Casos de Teste

Modelos de linguagem possuem limites rigorosos de janela de contexto e o custo de execução via API cresce proporcionalmente ao volume de dados fornecidos. Por isso, em vez de sobrecarregar o agente com tabelas gigantes de 1000 linhas, nossa abordagem para gerar os **dez casos de teste** concentra-se na **complexidade lógica** das instruções.

Esta etapa cumpre os seguintes requisitos da avaliação:
* **Tarefa Verificável:** Cada caso de teste possui um resultado matemático exato esperado (ex: um saldo final específico), permitindo que a verificação seja feita por um programa que dá a palavra final de forma objetiva, validando se o teste foi aprovado ou reprovado.
* **Limites do Agente:** Pelo menos três testes são desenhados para serem suficientemente desafiadores. Eles inserem comandos "bagunçados", estornos e cálculos de porcentagem no mesmo *prompt*, exigindo múltiplas chamadas de ferramentas e o uso intensivo da memória para alteração retrospectiva do estado financeiro.

Abaixo, definimos a função de verificação usada pelo laço (*harness*); a bateria completa de testes está na seção 5. **[CORREÇÃO 13]**

In [ ]:

# Função Auxiliar de Verificação (Harness)
# Esta função será usada pelo laço do agente para avaliar objetivamente se a saída estruturada
# retornada bate com a matemática exata exigida pelo caso de teste.
def verificar_resultado_teste(estado_obtido: dict, estado_esperado: dict) -> tuple[bool, str]:
    """
    Compara o estado financeiro devolvido pelo agente com o gabarito matemático do teste.
    Devolve um booleano de aprovação e uma mensagem detalhada para análise.
    """
    for chave, valor_esperado in estado_esperado.items():
        valor_obtido = estado_obtido.get(chave)

        # Tolera pequenas diferenças de arredondamento em ponto flutuante
        if valor_obtido is None or abs(float(valor_obtido) - float(valor_esperado)) > 0.01:
            return False, f"Falha na chave '{chave}'. Esperado: {valor_esperado}, Obtido: {valor_obtido}"

    return True, "Aprovado: Todos os valores matemáticos coincidem perfeitamente."

## 5. Validação e Testes

O laço de avaliação (*harness*) é o código em volta do modelo responsável por coordenar a execução das tarefas e julgar os resultados sem intervenção manual. Nesta etapa, consolidamos a **bateria de dez casos de teste** exigida para validar o comportamento do agente financeiro.

A suíte de testes foi estruturada para cobrir o escopo completo da tarefa verificável:
*   **Progressão de Dificuldade:** Os testes iniciais validam operações unitárias (adicionar uma receita, registrar um gasto).
*   **Exploração de Limites:** Pelo menos três testes (os finais) são propositalmente desafiadores. Eles inserem comandos ambíguos, estornos retrospectivos e cálculos de juros que forçam o agente a usar múltiplas ferramentas (como consultar o histórico para achar um valor, calcular o juros e depois registrar) em múltiplas etapas.
*   **Formatação e Métricas:** A rotina de testes iterará sobre os casos, executando o agente a partir de um estado limpo. Para cada execução, o laço imprimirá na tela a entrada, o resultado esperado, o resultado obtido e se o teste foi aprovado ou reprovado. Ao final, a taxa de acerto global será calculada e apresentada junto a uma análise de erros.

In [ ]:

# 1. Bateria Completa de 10 Casos de Teste
#  Quando "entrada" é uma lista, cada item é um turno separado, sem limpar a memória entre eles.
testes_financeiros = [
    # --- Testes Básicos (1 ferramenta, 1-2 etapas) ---
    {"id": 1, "entrada": "Recebi 1500 de salário hoje.", "esperado": {"saldo_final": 1500.0}},
    {"id": 2, "entrada": "Gastei 50 reais na padaria (Alimentação). Qual meu saldo, considerando que eu não tinha nada?", "esperado": {"saldo_final": -50.0}},
    {"id": 3, "entrada": "Tinha 0. Recebi 200 de um freela e gastei 80 no Uber (Transporte). Saldo final?", "esperado": {"saldo_final": 120.0}},
    {"id": 4, "entrada": "Adicione 100 de mesada e 50 de venda de um livro. Saldo?", "esperado": {"saldo_final": 150.0}},

    # --- Testes Intermediários (Uso de memória e múltiplas categorias) ---
    {"id": 5, "entrada": "Recebi 3000 de salário. Paguei 1500 de aluguel e 300 de mercado. Qual o total gasto e o saldo?", "esperado": {"saldo_final": 1200.0}},
    {"id": 6, "entrada": ["Gastei 100 de farmácia ontem.", "Hoje o estorno de 100 da farmácia caiu na conta. Qual o saldo atual?"], "esperado": {"saldo_final": 0.0, "total_gastos": 0.0}},
    {"id": 7, "entrada": "Recebi 500. Gastei 100 em Alimentação, 50 em Transporte e 200 em Alimentação de novo. Qual o total da categoria Alimentação?", "esperado": {"total_alimentacao": 300.0}},

    # --- Testes Desafiadores / Limites (Múltiplas ferramentas, cálculos, estornos e agregações) ---
    {"id": 8, "entrada": "Recebi 2000. Paguei a conta de luz atrasada de 150 com 10% de multa. Depois, um amigo me pagou 50 que me devia. Saldo final?", "esperado": {"saldo_final": 1885.0}},
    {"id": 9, "entrada": ["Ontem recebi 1000 e gastei 200 com Uber.", "Hoje a corrida do Uber foi cancelada e me estornaram os 200. Depois comprei um lanche de 40. Qual o saldo e o gasto com Transporte?"], "esperado": {"saldo_final": 960.0, "total_transporte": 0.0}},
    {"id": 10, "entrada": "Meu saldo inicial era 0. Recebi 5000 de salário. Gastei 2000 de aluguel. A imobiliária me devolveu 500 da caução do apartamento antigo (não é estorno do aluguel). Paguei 300 de mercado, mas devolvi um item e estornaram 50. Calcule o saldo final e me diga quanto gastei no total (sem contar o que foi estornado).", "esperado": {"saldo_final": 3250.0, "total_gastos": 2250.0}},

    # --- Testes Negativos / Tratamento de Exceções Lógicas ---
    # Testa a violação da regra de categorias permitidas
    {"id": 11, "entrada": "Gastei 150 em Ingressos de Show (Lazer).", "esperado": {"saldo_final": 0.0, "total_gastos": 0.0}},

    # Testa um estorno inválido (tentar estornar mais do que o valor original)
    {"id": 12, "entrada": ["Gastei 50 na padaria (Alimentação).", "Estorne 100 da padaria, por favor."], "esperado": {"saldo_final": -50.0, "total_gastos": 50.0}},

    # Testa alucinação do modelo perante ambiguidade extrema (falta do valor)
    {"id": 13, "entrada": "Comprei um lanche na padaria agora à tarde. Atualiza o saldo pra mim.", "esperado": {"saldo_final": 0.0, "total_gastos": 0.0}}
]

# 2. Laço de Execução e Verificação (Harness)
import time

def executar_bateria_de_testes(agente, testes):
    """
    Executa os testes, limpa o estado a cada iteração, avalia os resultados e formata a saída.
    Possui sistema de tentativas automáticas para contornar limites (429) da API gratuita.
    """
    testes_aprovados = 0
    total_testes = len(testes)

    print("="*50)
    print("INÍCIO DA VALIDAÇÃO DO AGENTE FINANCEIRO")
    print("="*50)

    for teste in testes:
        print(f"\n[ TESTE {teste['id']} ]")
        print(f"Entrada: {teste['entrada']}")
        print(f"Resultado Esperado: {teste['esperado']}")

        sucesso_execucao = False
        tentativas = 0
        max_tentativas = 5

        # Laço de repetição: tenta até dar certo ou exceder o limite de tentativas do teste atual
        while not sucesso_execucao and tentativas < max_tentativas:

            # O reset deve ficar DENTRO do while para limpar qualquer "sujeira"
            # na memória se a API tiver falhado a meio do teste na tentativa anterior
            resetar_memoria()

            try:
                entradas = teste["entrada"] if isinstance(teste["entrada"], list) else [teste["entrada"]]

                # Executa o agente
                mensagens_resultado = []
                for entrada in entradas:
                    historico = format_financial_state(finance_state)["content"]
                    mensagens_resultado = agente.run(build_context(FINANCIAL_SYSTEM, historico, entrada))

                # Extrai a saída estruturada
                resultado_obtido = extrair_relatorio(mensagens_resultado)

                print(f"Resposta do Agente: {mensagens_resultado[-1]['content']}")
                print(f"Resultado Obtido: {resultado_obtido}")

                # Verificação objetiva
                aprovado, motivo = verificar_resultado_teste(resultado_obtido, teste['esperado'])

                if aprovado:
                    print("Status: ✅ APROVADO")
                    testes_aprovados += 1
                else:
                    print(f"Status: ❌ REPROVADO ({motivo})")

                # Se chegou aqui, não houve erro de API. Sai do laço while e passa ao próximo teste.
                sucesso_execucao = True

            except Exception as e:
                erro_str = str(e)
                # Verifica se é erro de cota/limite da API
                if "429" in erro_str or "Quota" in erro_str or "Too Many Requests" in erro_str:
                    espera = 60 # Espera um minuto inteiro para a cota renovar com folga
                    print(f"⚠️ Limite da API atingido. Aguardando {espera} segundos para tentar o Teste {teste['id']} de novo... (Tentativa {tentativas+1}/{max_tentativas})")
                    time.sleep(espera)
                    tentativas += 1
                else:
                    # Se for outro erro de código, reprova imediatamente e quebra o while
                    print(f"Resultado Obtido: ERRO DE EXECUÇÃO ({erro_str})")
                    print("Status: ❌ REPROVADO (Falha no Agente)")
                    break

        if not sucesso_execucao and tentativas == max_tentativas:
            print("Status: ❌ REPROVADO (Cancelado após várias falhas de comunicação com a API)")

        # Pausa leve de 5 segundos entre testes diferentes (que deram certo) para não engasgar a API no teste seguinte
        time.sleep(5)

    # Cálculo de Métricas e Análise
    taxa_acerto = (testes_aprovados / total_testes) * 100

    print("\n" + "="*50)
    print("RESUMO DA AVALIAÇÃO")
    print("="*50)
    print(f"Taxa de Acerto: {taxa_acerto:.1f}% ({testes_aprovados}/{total_testes})")
    # [O texto provisório da análise de erros saiu daqui: a análise fica na célula de texto seguinte.

# Execução da bateria com o agente real
executar_bateria_de_testes(agent, testes_financeiros)

INÍCIO DA VALIDAÇÃO DO AGENTE FINANCEIRO

[ TESTE 1 ]
Entrada: Recebi 1500 de salário hoje.
Resultado Esperado: {'saldo_final': 1500.0}
Resposta do Agente: Saldo: R$ 1.500,00  
Total de gastos: R$ 0,00  
Nenhum gasto registrado por categoria.
Resultado Obtido: {'saldo_final': 1500.0, 'total_gastos': 0.0, 'total_alimentacao': None, 'total_transporte': None}
Status: ✅ APROVADO

[ TESTE 2 ]
Entrada: Gastei 50 reais na padaria (Alimentação). Qual meu saldo, considerando que eu não tinha nada?
Resultado Esperado: {'saldo_final': -50.0}
Resposta do Agente: Saldo: -R$ 50,00.
Resultado Obtido: {'saldo_final': -50.0, 'total_gastos': 50.0, 'total_alimentacao': 50.0, 'total_transporte': None}
Status: ✅ APROVADO

[ TESTE 3 ]
Entrada: Tinha 0. Recebi 200 de um freela e gastei 80 no Uber (Transporte). Saldo final?
Resultado Esperado: {'saldo_final': 120.0}
Resposta do Agente: Saldo final: **R$ 120,00**.
Resultado Obtido: {'saldo_final': 120.0, 'total_gastos': 80.0, 'total_alimentacao': None, 'total_

### 6. Breve Análise dos Principais Erros Observados

Após a execução da bateria expandida de testes, o agente obteve uma **taxa de acerto de 76,9% (10/13)**. O modelo demonstrou um desempenho perfeito nos casos de simples e lógicas complexas, mas as falhas introduzidas nos testes negativos revelaram comportamentos interessantes sobre a autonomia do LLM e os limites da infraestrutura.

Abaixo está o detalhamento do comportamento do agente:

**1. Sucesso Absoluto na Lógica Financeira (Testes 1 a 10):**
O agente não cometeu nenhum erro de cálculo, manipulação de estado ou interpretação. Ele lidou perfeitamente com a aplicação de multas percentuais (Teste 8), utilizou a memória de curto prazo para identificar transações em turnos anteriores (Testes 6 e 9) e diferenciou corretamente estornos de falsas entradas (Teste 10).

**2. O Comportamento de "Acomodação" do LLM (Falha no Teste 11):**

* **Ocorrência:** O utilizador tentou registar um gasto na categoria "Lazer" (não permitida). O resultado esperado era que a ferramenta recusasse a ação.
* **Análise:** Em vez de retornar um erro rígido e travar a operação, o LLM demonstrou um comportamento proativo: ele interpretou que "Lazer" não existia, mas reclassificou autonomamente a despesa para a categoria genérica permitida ("Outros") e processou o gasto. Isso alterou o saldo para -150.0, reprovando o teste. Isso evidencia a tendência do modelo de tentar completar a tarefa e ser "útil", mesmo quando as instruções exigem restrição estrita.

**3. Gargalos de Infraestrutura da API (Falha no Teste 12):**

* **Ocorrência:** O teste foi cancelado e reprovado após esgotar o limite máximo de 5 tentativas.
* **Análise:** Esta não foi uma falha lógica do agente. O ambiente atingiu o limite de requisições por minuto (*Rate Limit* - HTTP 429) da API gratuita. Apesar do mecanismo de repetição (*harness*) ter tentado contornar o erro com pausas de 60 segundos, o bloqueio do provedor persistiu tempo suficiente para estourar o limite de tentativas.

**4. Falso Negativo por Incompatibilidade de Extração (Falha no Teste 13):**

* **Ocorrência:** O teste reprovou porque o dicionário obteve `None` em vez de `0.0` para o total de gastos.
* **Análise:** O agente, na verdade, **acertou a lógica**. Ao receber a instrução ambígua "Comprei um lanche", ele recusou-se a inventar (*alucinar*) um valor e perguntou: *"Poderia informar o valor do lanche?"*. O erro ocorreu estritamente na camada de código de avaliação: como nenhuma ferramenta foi ativada, o relatório gerado pelo Pydantic deixou a chave `total_gastos` vazia (`None`), o que o verificador matemático considerou diferente de `0.0`.